# TensorCore-FPGA Demo
## PYNQ-Z1 Matrix Multiplication Accelerator

This notebook demonstrates the TensorCore systolic array accelerator on PYNQ.

In [ ]:
from pynq import Overlay, allocate
import numpy as np
import time

# Load overlay
ol = Overlay('tensorcore.bit')
tc = ol.tensorcore_0
print(f'TensorCore loaded: {tc}')

## 1. Simple Matrix Multiplication

In [ ]:
# Create test matrices (4x4)
A = np.random.randint(-128, 127, (4, 4), dtype=np.int8)
B = np.random.randint(-128, 127, (4, 4), dtype=np.int8)

print('Matrix A:')
print(A)
print('\nMatrix B:')
print(B)

In [ ]:
# Allocate DMA buffers
in_buf = allocate(shape=(32,), dtype=np.int8)
out_buf = allocate(shape=(16,), dtype=np.int8)

# Pack matrices
in_buf[:16] = A.flatten()
in_buf[16:32] = B.flatten()

# Run accelerator
start = time.time()
tc.write(0x00, 1)  # Start
while tc.read(0x00) & 0x2 == 0:  # Wait for done
    pass
elapsed = time.time() - start

# Get result
C_hw = out_buf[:16].reshape(4, 4)
print(f'Hardware result (elapsed: {elapsed*1e6:.1f} us):')
print(C_hw)

In [ ]:
# Verify with NumPy
C_sw = np.matmul(A.astype(np.int16), B.astype(np.int16)).astype(np.int8)
print('Software reference:')
print(C_sw)
print(f'\nMatch: {np.allclose(C_hw, C_sw)}')

## 2. Performance Benchmark

In [ ]:
# Benchmark 1000 iterations
iterations = 1000
start = time.time()
for _ in range(iterations):
    tc.write(0x00, 1)
    while tc.read(0x00) & 0x2 == 0:
        pass
total = time.time() - start

ops_per_iter = 4 * 4 * 4 * 2  # M*N*K MACs = 2 ops each
total_ops = ops_per_iter * iterations
gops = total_ops / total / 1e9

print(f'Iterations: {iterations}')
print(f'Total time: {total:.3f} s')
print(f'Throughput: {gops:.2f} GOPS')